Q1. How many lesson pages

In [1]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

print(len(documents))

72


Q2. Indexing and searching

In [2]:
index = build_index(documents)

In [3]:
def index_search(question):
    search_results = index.search(
        question,
        num_results=5
    )

    return search_results

In [4]:
question = 'How does the agentic loop keep calling the model until it stops?'
search_results = index_search(question)
search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

Q3. RAG

In [10]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. 

If the answer is not found in the context, respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

response, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(response)

It keeps calling the model in a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call` items.
3. Run those tools and append the results to the messages.
4. If there were no function calls, `break` out of the loop.

So the loop stops when `has_function_calls == False`, meaning the model returned a final answer with no more tool calls.


In [11]:
print('Input tokens: ', input_tokens)

Input tokens:  7043


Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

Q5. RAG with chunking

In [17]:
chunk_index = build_index(chunks)

chunk_assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
)

response, input_tokens = chunk_assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(response)

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- if there is at least one function call, it runs the tool, adds the result to `messages`, and continues
- if there are no function calls in that turn, it breaks out of the loop

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words: the agent keeps looping until the model returns a final answer with no more tool calls.


In [18]:
print('Input tokens: ', input_tokens)

Input tokens:  2227


In [19]:
7043/2227

3.162550516389762

Q6. Turning it into an agent

In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [35]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
question = 'How does the agentic loop work, and how is it different from plain RAG?'

In [42]:
def search(query: str) -> dict[str, str]:
    print(len(chunk_index))

    return chunk_index.search(
        query,
        num_results=5,
    )

In [43]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [44]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [45]:
chat_interface = IPythonChatInterface()
callback=DisplayingRunnerCallback(chat_interface)

In [46]:
runner = OpenAIResponsesRunner(
    tools=agent_tools, 
    developer_prompt=instructions, 
    chat_interface=chat_interface, 
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [47]:
result = runner.loop(
    prompt=question, 
    callback=callback
)

-> Response received


-> Response received


-> Response received
